In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

from catboost import CatBoostClassifier
import matplotlib.pyplot as plt


In [ ]:

# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:

# 1) Read the dataset
# List files (already confirmed)
files = os.listdir(path)
print("Files in dataset folder:", files)

# Read the dataset using the correct path
csv_path = os.path.join(path, "Q3_data.csv")
df = pd.read_csv(csv_path)

In [ ]:
# 2) Inspect the first few rows
df.head()                         # Display the first 5 rows of the dataset


In [ ]:
# Dataset info
df.info()


In [ ]:
df.describe()

In [ ]:
# Separate features and target
target_col = "Target"
X = df.drop(columns=[target_col])
y = df[target_col]

# Check missing values
print("Missing values before:", X.isnull().sum().sum())

# Fill missing values using median (robust for anonymized numeric data)
X = X.fillna(X.median())

print("Missing values after:", X.isnull().sum().sum())


In [ ]:
# Check duplicates
duplicates = X.duplicated().sum()
print("Number of duplicate rows:", duplicates)

# Remove duplicates if any
if duplicates > 0:
    X = X.drop_duplicates()
    y = y.loc[X.index]


In [ ]:
# Check if categorical variables exist
cat_cols = X.select_dtypes(include=["object", "category"]).columns

if len(cat_cols) == 0:
    print("No categorical variables were found so encoding is not required.")
else:
    print("Categorical columns found:", list(cat_cols))
    # If needed you could apply encoding here like one-hot encoding)


In [ ]:
# Apply feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled feature shape:", X_scaled.shape)


In [ ]:
# Check target
print("Target value counts:")
print(y.value_counts())

print("\nTarget distribution (normalized):")
print(y.value_counts(normalize=True))

In [ ]:
target_col = "Target"
X = df.drop(columns=[target_col])
y = df[target_col]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
# 2) StratifiedKFold (classification + imbalance)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X_scaled, y), start=1):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # 3) Train CatBoostClassifier
    model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        random_seed=42,
        verbose=0
    )

    model.fit(X_train, y_train)

    # 4) Evaluate using F1 Score ONLY
    y_pred = model.predict(X_test)
    f1 = f1_score(y_test, y_pred)
    scores.append(f1)

    print(f"Fold {fold} F1: {f1:.4f}")

# 5) Print averaged score
print("-" * 30)
print(f"Average F1 across all folds: {np.mean(scores):.4f}")

In [ ]:
importances = model.get_feature_importance()

# Create DataFrame for feature importance
feat_imp_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)


In [ ]:
# Plot top 20 important features
plt.figure(figsize=(10,6))
plt.barh(feat_imp_df["Feature"].head(20)[::-1],
         feat_imp_df["Importance"].head(20)[::-1])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top Feature Importances (CatBoost)")
plt.show()

In [ ]:
# Task Bonus: Write your code here: